## Initialization

In [5]:
from torch.func import vjp
import torch
from torch.func import jacrev, functional_call
import torch.nn as nn
from torch import Tensor

import torch.nn.functional as F

import sys
import os

current_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root_dir = os.path.abspath(os.path.join(current_notebook_dir, '../../'))

# 将这个父目录添加到sys.path的最前面
if project_root_dir not in sys.path:
    sys.path.insert(0, project_root_dir)

print(sys.path)

['/home/hqdeng7/lijuyang/generalization', '/home/hqdeng7/.conda/envs/ljy/lib/python311.zip', '/home/hqdeng7/.conda/envs/ljy/lib/python3.11', '/home/hqdeng7/.conda/envs/ljy/lib/python3.11/lib-dynload', '', '/home/hqdeng7/.conda/envs/ljy/lib/python3.11/site-packages']


In [10]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans, KMeans
import matplotlib.pyplot as plt
from sklearn.cluster import Birch
from sklearn.preprocessing import StandardScaler

In [6]:
from loss_distribution.pytorch_script.visual_utils \
	import load_cifar10_data, load_model_state_dict

from ntk_result.trials.utils import *

import torchvision
import torchvision.transforms as transforms

data_pth = '/home/hqdeng7/lijuyang/generalization/loss_distribution/pytorch_script/data/cifar10'
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
train_ds = torchvision.datasets.CIFAR10(root=data_pth, train=True, download=True, transform=transform)
test_ds = torchvision.datasets.CIFAR10(root=data_pth, train=False, download=True, transform=transform)

In [7]:
model200_path = '/home/hqdeng7/lijuyang/generalization/loss_distribution/model_training_results/cifar10_resnet20/model_200.pth'
model200 = load_model_state_dict('cifar10', 'resnet20', 10, model200_path, 'cuda')
model100_path = '/home/hqdeng7/lijuyang/generalization/loss_distribution/model_training_results/cifar10_resnet20/model_100.pth'
model100 = load_model_state_dict('cifar10', 'resnet20', 10, model100_path, 'cuda')

  从字典中提取模型状态字典...
提取成功
  从字典中提取模型状态字典...
提取成功


## Compute all grads

In [5]:
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.func import functional_call, jacrev

# 假设您已经定义了 compute_param_gradients 和 flatten_grads_dict
# ...

# def get_all_gradients(model: nn.Module, ds: Dataset, batch_size: int = 32, device='cuda'):
#     model.to(device)
#     model.eval()

#     dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
#     all_grads = []

#     for inputs, labels in tqdm(dl):
#         grads_per_sample_dict = compute_param_gradients(model, inputs, labels, device=device)
        
#         flattened_grad = flatten_grads_dict(grads_per_sample_dict)
#         all_grads.append(flattened_grad)
#         torch.cuda.empty_cache()

#     all_grads_tensor = torch.cat(all_grads, dim=0)

#     return all_grads_tensor

In [6]:
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.func import functional_call, jacrev

# def compute_all_grads(model: nn.Module, ds: Dataset, device='cuda', batch_size: int = 64):
#     """
#     使用 torch.func.vmap 向量化地获取所有样本的梯度。
#     """
#     model.to(device)
#     model.eval()

#     params = dict(model.named_parameters())
#     buffers = dict(model.named_buffers())

#     dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

#     # 将 compute_single_gradient 定义为内部函数，以便访问 model
#     def compute_single_gradient(p, b, i, l):
#         outputs = functional_call(model, (p, b), i.unsqueeze(0))
#         losses = nn.CrossEntropyLoss(reduction='none')(outputs, l.unsqueeze(0))
#         return losses

#     # vmap 向量化 jacrev，用于批量计算
#     grads_fn = jacrev(compute_single_gradient, argnums=0)
#     vmap_grads_fn = torch.func.vmap(grads_fn, in_dims=(None, None, 0, 0))

#     all_grads = []

#     for inputs, labels in tqdm(dl):
#         inputs, labels = inputs.to(device), labels.to(device)
        
#         # 使用 vmap_grads_fn 进行高效的批量梯度计算
#         grads_per_sample_dict = vmap_grads_fn(params, buffers, inputs, labels)
        
#         flattened_grad = flatten_grads_dict(grads_per_sample_dict)
#         all_grads.append(flattened_grad)
        
#         # 释放缓存以防万一
#         torch.cuda.empty_cache()

#     all_grads_tensor = torch.cat(all_grads, dim=0)

#     return all_grads_tensor

def compute_all_grads(model: nn.Module, ds: Dataset, device='cuda', batch_size: int = 64):
    """
    使用 torch.func.vmap 向量化地获取所有样本的梯度。
    """

    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

    all_grads = []

    for inputs, labels in tqdm(dl):
        all_grads.append(compute_param_grads(model100, inputs, labels))
    all_grads_tensor = torch.cat(all_grads, dim=0)

    return all_grads_tensor

In [7]:
all_grads = compute_all_grads(model100, test_ds)

100%|██████████| 157/157 [00:10<00:00, 15.39it/s]


In [8]:
all_grads.requires_grad

False

## high test loss cluster

### cluster in test hloss samples

In [18]:
model100_losses_fn = get_batch_loss_fn(model100)
hloss_samples = {}
hloss_samples[('test', 'epoch100', 'k500')] =find_topk_samples(test_ds, fn=model100_losses_fn, k=500)

100%|██████████| 40/40 [00:03<00:00, 13.32it/s]


In [19]:
hloss_samples_losses = {key: np.array(list(zip(*value))[0]) for key, value in hloss_samples.items()}
hloss_samples_indices = {key: np.array(list(zip(*value))[1]) for key, value in hloss_samples.items()}	

In [ ]:
from torch.utils.data import Subset
hloss_sample_grads = {}
hloss_sample_grads[('test', 'epoch100', 'k500')] = \
	compute_all_grads(model100, Subset(test_ds, hloss_samples_indices[('test', 'epoch100', 'k500')]))

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(hloss_sample_grads[('test', 'epoch100', 'k500')].tolist())

In [11]:
def dim_reduc_cluster(X, n_clusters: int, n_components: int = None):
	if n_components != None:
		print("\n使用PCA进行降维...")
		pca = PCA(n_components=0.5, random_state=42)
		X_pca = pca.fit_transform(X)
		print(f"降维后的数据形状: {X_pca.shape}")
	else:
		X_pca = X

	# MiniBatchKMeans比传统的KMeans更适合处理大规模数据，因为它每次只使用一小部分数据进行更新
	print("\n使用MiniBatchKMeans进行聚类...")
	n_clusters = n_clusters
	kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
	kmeans.fit(X_pca)

	# 获取聚类结果
	labels = kmeans.labels_
	print("\n聚类完成！")
	print(f"每个聚类的样本数: {[np.sum(labels == i) for i in range(n_clusters)]}")

	return labels if n_components == None else (X_pca, labels)

In [12]:
X_pca = np.load('X_pca.npy')
cls30_labels = dim_reduc_cluster(X_pca, 30)


使用MiniBatchKMeans进行聚类...

聚类完成！
每个聚类的样本数: [37, 13, 32, 24, 20, 36, 17, 12, 9, 17, 14, 13, 15, 17, 31, 9, 20, 14, 22, 35, 20, 5, 24, 14, 4, 1, 7, 3, 9, 6]


### visualization

In [ ]:
def plot_cluster(labels, cluster_idx, len=4):
	fig, axes = plt.subplots(len, len, figsize=(16, 16))
	axes = axes.flatten()
	cluster_sample_indices = hloss_samples_indices[('test', 'epoch100', 'k500')][np.array(labels) == cluster_idx]
	for i in range(len*len):
		img = inverse_trans_cifar10(test_ds[cluster_sample_indices[i]][0])
		img_show(img, ax=axes[i])		


In [ ]:
plot_cluster(cls15_labels, 0, len=5)

In [ ]:
plot_cluster(cls15_labels, 1, len=5)

In [ ]:
plot_cluster(cls15_labels, 2, len=4)

In [ ]:
plot_cluster(cls15_labels, 3, len=4)

In [ ]:
plot_cluster(cls15_labels, 14, len=7)

In [ ]:
plot_cluster(cls15_labels, 13, len=5)

In [ ]:

cls30_labels = dim_reduc_cluster(X_pca, 30)

In [ ]:
plot_cluster(cls30_labels, 0, len=7)

In [ ]:
plot_cluster(cls30_labels, 15, len=5)

In [ ]:
plot_cluster(cls30_labels, 13)

In [ ]:
plot_cluster(cls30_labels, 17, len=3)

In [ ]:
plot_cluster(cls30_labels, 27, len=4)

In [ ]:
plot_cluster(cls30_labels, 28, len=4)

### peek the grad dict

In [ ]:
def compute_param_grads_dict(model: nn.Module, 
						inputs: torch.Tensor,
						labels: torch.Tensor,
						loss_fn = nn.CrossEntropyLoss(reduction='mean'),
						device='cuda'):
	"""
	使用 torch.func.vmap 向量化地获取所有样本的梯度。
	"""
	loss_fn = loss_fn if loss_fn.reduction == 'mean' else type(loss_fn)(reduction='mean')

	model.to(device)
	model.eval()

	params = dict(model.named_parameters())
	buffers = dict(model.named_buffers())

	# 将 compute_single_gradient 定义为内部函数，以便访问 model
	def compute_single_gradient(params, buffers, input, label):
		output = functional_call(model, (params, buffers), input.unsqueeze(0))
		loss = loss_fn(output, label.unsqueeze(0))
		return loss

	# vmap 向量化 jacrev，用于批量计算
	grads_fn = torch.func.grad(compute_single_gradient, argnums=0)
	vmap_grads_fn = torch.func.vmap(grads_fn, in_dims=(None, None, 0, 0))

	inputs, labels = inputs.to(device), labels.to(device)
	
	# 使用 vmap_grads_fn 进行高效的批量梯度计算
	grads_per_sample_dict = vmap_grads_fn(params, buffers, inputs, labels)
	
	# 释放缓存以防万一
	torch.cuda.empty_cache()

	return grads_per_sample_dict

In [ ]:
input, label = test_ds[hloss_samples_indices[('test', 'epoch100', 'k500')][2]]
input_, label_ = input.unsqueeze(0), torch.tensor(label).unsqueeze(0)
compute_param_grads_dict(model100, input_, label_)

In [ ]:
hloss_samples_indices[('test', 'epoch100', 'k500')]

### clusters internal cos sim

In [ ]:
cls30_indices_arrs = [hloss_samples_indices[('test', 'epoch100', 'k500')][cls30_labels == i] for i in range(30)]


In [ ]:
def compute_single_gradient(model: nn.Module, input, label, loss_fn=nn.CrossEntropyLoss(reduction='mean')):
	input_ = input.unsqueeze(0)
	label_ = torch.tensor(label).unsqueeze(0)
	return compute_param_grads(model, input_, label_, loss_fn).squeeze(0)

In [ ]:
min_i = 0
min_j = 0
min_cos_sim = 1
tar_arr = cls30_indices_arrs[0]

for i in tqdm(range(len(tar_arr))):
	for j in range(i+1, len(tar_arr)):
		input1, label1 = test_ds[tar_arr[i]]
		input2, label2 = test_ds[tar_arr[j]]
		grad1 = compute_single_gradient(model100, input1, label1)
		grad2 = compute_single_gradient(model100, input2, label2)
		cos_sim = F.cosine_similarity(grad1, grad2, dim=0)
		if cos_sim < min_cos_sim:
			min_cos_sim = cos_sim
			min_i, min_j = i, j

print(min_i, min_j, min_cos_sim)

In [ ]:
input1, label1 = test_ds[tar_arr[min_i]]
input2, label2 = test_ds[cls30_indices_arrs[1][27]]
grad1 = compute_single_gradient(model100, input1, label1)
grad2 = compute_single_gradient(model100, input2, label2)
cos_sim = F.cosine_similarity(grad1, grad2, dim=0)
cos_sim

In [ ]:
min_i = 0
min_j = 0
min_cos_sim = 1
tar_arr = cls30_indices_arrs[15]

for i in tqdm(range(len(tar_arr))):
	for j in range(i+1, len(tar_arr)):
		input1, label1 = test_ds[tar_arr[i]]
		input2, label2 = test_ds[tar_arr[j]]
		grad1 = compute_single_gradient(model100, input1, label1)
		grad2 = compute_single_gradient(model100, input2, label2)
		cos_sim = F.cosine_similarity(grad1, grad2, dim=0)
		if cos_sim < min_cos_sim:
			min_cos_sim = cos_sim
			min_i, min_j = i, j

print(min_i, min_j, min_cos_sim)

## training clusters from test

### preparation

In [18]:
cls30_indices_arrs = [hloss_samples_indices[('test', 'epoch100', 'k500')][cls30_labels == i] for i in range(30)]

NameError: name 'hloss_samples_indices' is not defined

### using first one to cluster

In [ ]:
tar_idx = cls30_indices_arrs[0][0]
ref_input, ref_label = test_ds[tar_idx]
fn = get_batch_grad_cos_fn(model100, ref_input, ref_label)
top500_train_cos_sim = find_topk_samples(train_ds, fn, k=500)

In [ ]:
test_indices = cls30_indices_arrs[0]
train_indices = list(zip(*top500_train_cos_sim))[1]

cos_sims_arr = np.array([])
for test_idx in tqdm(test_indices):
	input, label = test_ds[test_idx]
	batch_grad_cos_fn = get_batch_grad_cos_fn(model100, input, label)

	train_top500_ds = Subset(train_ds, train_indices)
	train_top500_dl = DataLoader(train_top500_ds, 512)
	for batch in train_top500_dl:
		cos_sims = batch_grad_cos_fn(batch)

	cos_sims_arr = np.append(cos_sims_arr, cos_sims)

In [ ]:
cos_sims_arr = np.reshape(cos_sims_arr, (len(test_indices), -1))

In [ ]:
print('逐个平均cos sim')
print(np.round(np.mean(cos_sims_arr, axis=1), 2))

In [ ]:
print('整体平均cos sim')
np.mean(cos_sims_arr)

In [ ]:
test_idx = cls30_indices_arrs[0][0]
input, label = test_ds[test_idx]

# 根据该测试样本label筛选训练集对应子集
target_label = label
target_label_indices = [i for i, (_, lbl) in enumerate(train_ds) if lbl == target_label]

# 构造训练集子集和 DataLoader
train_subset = Subset(train_ds, target_label_indices)
train_dl = DataLoader(train_subset, batch_size=512, shuffle=False)

# 得到用于计算相似度的函数
batch_grad_cos_fn = get_batch_grad_cos_fn(model100, input, label)

# 收集相似度
cos_sims_list = []
for batch in tqdm(train_dl):
    batch_sim = batch_grad_cos_fn(batch)  # 返回 numpy 数组或 tensor
    # 如果是 tensor，先转 numpy
    if hasattr(batch_sim, 'cpu'):
        batch_sim = batch_sim.cpu().numpy()
    cos_sims_list.append(batch_sim)

# 拼接成一个大数组
cos_sims = np.concatenate(cos_sims_list)

# 计算均值

In [ ]:
print("训练集样本同类别cos sim:", np.round(cos_sims.mean(), 2))

In [ ]:
fig, axes = plt.subplots(16, 8, figsize=(32, 64))
axes = axes.flatten()
for i in range(128):
	input = train_ds[train_indices[i]][0]
	img = inverse_trans_cifar10(input)
	img_show(img, ax=axes[i])

### using centroid to cluster

In [9]:
'''
 assume existing:
		X_pca
		hloss_samples_indices

'''

from scipy.spatial.distance import cdist
def cluster_and_get_centroids(data, n_clusters: int):
	kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
	kmeans.fit(data)

	return kmeans.labels_, kmeans.cluster_centers_


def find_closest_samples(samples, center):
	# 计算该聚类中所有样本到其中心点的距离
	distances = cdist(samples, [center], 'euclidean').flatten()
	
	# 找到最小距离及其对应的样本索引
	min_dist_idx = np.argmin(distances)
	
	return min_dist_idx


In [13]:
X_pca = np.load('X_pca.npy')
cls30_labels, cls30_centers = cluster_and_get_centroids(X_pca, 30)

In [14]:
len(cls30_labels[cls30_labels == 0])

37

In [21]:
test_indices = cls30_indices_arrs[0]
tar_idx = test_indices[find_closest_samples(X_pca[cls30_labels == 0], cls30_centers[0])]

In [22]:
ref_input, ref_label = test_ds[tar_idx]
fn = get_batch_grad_cos_fn(model100, ref_input, ref_label)
top500_train_center = find_topk_samples(train_ds, fn, k=500)

100%|██████████| 196/196 [00:30<00:00,  6.42it/s]


In [ ]:
def compute_test_train_cos_sims_arr(test_indices, train_indices):
	train_top500_ds = Subset(train_ds, train_indices)
	train_top500_dl = DataLoader(train_top500_ds, 512)

	cos_sims_arr = np.array([])
	for test_idx in tqdm(test_indices):
		input, label = test_ds[test_idx]
		batch_grad_cos_fn = get_batch_grad_cos_fn(model100, input, label)

		for batch in train_top500_dl:
			cos_sims = batch_grad_cos_fn(batch)

		cos_sims_arr = np.append(cos_sims_arr, cos_sims)

	cos_sims_arr = np.reshape(cos_sims_arr, (len(test_indices), -1))
	return cos_sims_arr


In [ ]:
test_indices = cls30_indices_arrs[0]
train_indices = list(zip(*top500_train_center))[1]
cos_sims_arr = compute_test_train_cos_sims_arr(test_indices, train_indices)

In [ ]:
print('用距离聚类中心最近的样本')
print('逐个平均cos sim')
print(np.round(np.mean(cos_sims_arr, axis=1), 2))
print('整体平均cos sim')
print(np.mean(cos_sims_arr))

### all train indices arrs

In [ ]:
filtered_cls30_indices_arrs = [arr for arr in cls30_indices_arrs if len(arr) > 8]
len(filtered_cls30_indices_arrs)

In [ ]:
train_indices_arrs = []
for i in range(30):
	test_indices = cls30_indices_arrs[i]
	if len(test_indices) > 8:
		tar_idx = test_indices[find_closest_samples(X_pca[cls30_labels == i], cls30_centers[i])]
		ref_input, ref_label = test_ds[tar_idx]
		fn = get_batch_grad_cos_fn(model100, ref_input, ref_label)
		top500_train_center = find_topk_samples(train_ds, fn, k=500)
		train_indices = list(zip(*top500_train_center))[1]
		train_indices_arrs.append(train_indices)

train_indices_arrs = np.array(train_indices_arrs) 

In [ ]:
train_indices_arrs.shape

### other baselines

In [ ]:
test_indices = cls30_indices_arrs[0]
rand_train_indices = np.random.randint(0, 50000, size=500)
cos_sims_arr = compute_test_train_cos_sims_arr(test_indices, rand_train_indices)

print('逐个平均cos sim')
print(np.round(np.mean(cos_sims_arr, axis=1), 2))
print('整体平均cos sim')
print(np.mean(cos_sims_arr))

In [ ]:
np.round(np.mean(cos_sims_arr), 2)

In [ ]:
target_label_indices = [i for i, (_, lbl) in enumerate(train_ds) if lbl == target_label]

### 10 clusters result

In [ ]:
cls10_labels = dim_reduc_cluster(X_pca, 10)

In [ ]:
cls10_indices_arrs = \
	[hloss_samples_indices[('test', 'epoch100', 'k500')][cls10_labels == i] for i in range(10)]
cls10_labels, cls10_centers = cluster_and_get_centroids(X_pca, 10)

In [ ]:
test_indices = cls10_indices_arrs[0]
test_indices[find_closest_samples(X_pca[cls10_labels == 0], cls10_centers[0])]

In [ ]:
train_indices_arrs = []
for i in range(10):
	test_indices = cls10_indices_arrs[i]
	tar_idx = test_indices[find_closest_samples(X_pca[cls10_labels == i], cls10_centers[i])]
	ref_input, ref_label = test_ds[tar_idx]
	fn = get_batch_grad_cos_fn(model100, ref_input, ref_label)
	top500_train_center = find_topk_samples(train_ds, fn, k=500)
	train_indices = list(zip(*top500_train_center))[1]
	train_indices_arrs.append(train_indices)

cls10_train_indices_arrs = np.array(train_indices_arrs) 

## save results

In [ ]:
np.save('test_top500_e100_indices.npy', hloss_samples_indices['test', 'epoch100', 'k500'])

In [ ]:
np.savez_compressed('filtered_cls30_indices_arrs.npz', *filtered_cls30_indices_arrs)

In [ ]:
data = np.load("filtered_cls30_indices_arrs.npz", allow_pickle=True)
filtered_cls30_indices_arrs = [data[f"arr_{i}"] for i in range(len(data.files))]

In [ ]:
np.save('train_indices_arrs.npy', train_indices_arrs)

In [ ]:
np.save('X_pca.npy', X_pca)

In [ ]:
np.savez_compressed('cls10_indices_arrs.npz', *cls10_indices_arrs)
np.save('cls10_train_indices_arrs.npy', cls10_train_indices_arrs)

In [ ]:
train_indices_arrs[0]

## not to train

In [26]:
from scipy.spatial.distance import cdist
def cluster_and_get_centroids(data, n_clusters: int):
	kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
	kmeans.fit(data)

	return kmeans.labels_, kmeans.cluster_centers_


def find_closest_samples(samples, center):
	# 计算该聚类中所有样本到其中心点的距离
	distances = cdist(samples, [center], 'euclidean').flatten()
	
	# 找到最小距离及其对应的样本索引
	min_dist_idx = np.argmin(distances)
	
	return min_dist_idx

X_pca = np.load('X_pca.npy')
cls30_labels, cls30_centers = cluster_and_get_centroids(X_pca, 30)
cls30_indices_arrs = np.load('cls30_indices_arrs.npz', allow_pickle=True)
cls30_indices_arrs = [cls30_indices_arrs[f"arr{i}"] for i in range(len(cls30_indices_arrs.files))]

In [28]:
from torch.func import jacrev, functional_call, vmap

def compute_batch_logits(model, params, buffers, inputs):
	return functional_call(model, (params, buffers), inputs)


def get_param_jac(model: nn.Module, inputs: Tensor):
    """
    返回形状: (batch_size, output_dim, total_params)
    使用 vmap+jacrev 降低显存占用
    """
    params, buffers = dict(model.named_parameters()), dict(model.named_buffers())

    # 单样本 Jacobian
    def single_jac(x):
        jacs = jacrev(compute_batch_logits, argnums=1)(model, params, buffers, x.unsqueeze(0))
        # 拼接到一个 tensor
        jacob_views = [jac.reshape(1, jac.shape[1], -1) for jac in jacs.values()]
        return torch.cat(jacob_views, dim=2)  # (1, out_dim, P)

    # vmap 按 batch 并行
    jacob_all = vmap(single_jac)(inputs)  # (B, 1, out_dim, P)
    jacob_all = jacob_all.squeeze(1)      # (B, out_dim, P)

    return jacob_all


def get_batch_myentk_fn(model, 
						   ref_input,
						   ref_label,
						   loss_fn=nn.CrossEntropyLoss(reduction='none'), 
						   device='cuda'):
	model.eval()
	loss_fn = loss_fn if loss_fn.reduction == 'none' else type(loss_fn)(reduction='none')
	model = model.to(device)

	ref_input = ref_input.to(device).unsqueeze(0)

	ref_jacob = get_param_jac(model, ref_input)
	t = torch.full((10,), -1, device=device)
	t[ref_label] = 1
	left_mat = ref_jacob * t.view(1, 10, 1)	# left_mat: [1, out_dim, param_dim]
	left_mat = torch.sum(left_mat.squeeze(0), 0) # left_mat: [param_dim]

	def batch_myentk_fn(batch):
		inputs, labels = batch
		inputs, labels = inputs.to(device), labels.to(device)

		jacobs = get_param_jac(model, inputs)   # jacobs: [B, out_dim, param_dim]
		right_mat = jacobs[torch.arange(len(inputs)), labels]  # right_mat: [B, param_dim]

		results = torch.mv(right_mat, left_mat)
		return results
	
	return batch_myentk_fn


def find_topk_samples(
	ds, 
	fn,
	k=16,
	batch_size=256,
	reverse=False,
	show_progress=True
):
	
	dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

	heap = []
	idx_offset = 0

	iterator = tqdm(dl) if show_progress else dl
	for batch in iterator:

		batch_res = fn(batch)
		if reverse:
			# 取最小k个，先取负数，再存进堆
			batch_res_idx = [(-res, idx_offset+i) for i, res in enumerate(batch_res)]
		else:
			batch_res_idx = [(res, idx_offset+i) for i, res in enumerate(batch_res)]

		for res_idx in batch_res_idx:
			if len(heap) < k:
				heapq.heappush(heap, res_idx)
			else:
				heapq.heappushpop(heap, res_idx)
		
		idx_offset += len(batch[0])

	if reverse:
		# 负数还原
		heap = [(-res, idx) for res, idx in heap]

	heap.sort(key=lambda x: x[0], reverse=not reverse)

	return heap

tar_idx = test_indices[find_closest_samples(X_pca[cls30_labels == 0], cls30_centers[0])]
ref_input, ref_label = test_ds[tar_idx]
fn = get_batch_myentk_fn(model100, ref_input, ref_label)
not_train_indices = find_topk_samples(train_ds, fn, k=2, batch_size=64, reverse=True)

 18%|█▊        | 141/782 [00:14<01:06,  9.70it/s]


KeyboardInterrupt: 

In [17]:
from loss_distribution.pytorch_script.models import get_model
new_model = get_model('cifar10', 'resnet20', weights=None, num_classes=10)
new_model.load_state_dict(model100.state_dict())
new_model.to('cuda')
new_model.eval()

train_dl = DataLoader(train_ds, batch_size = 1)
inputs, labels = next(iter(train_dl))
inputs = inputs.to('cuda')
labels = labels.to('cuda')

jacobs = get_param_jac(new_model, inputs)
jacobs[torch.arange(len(inputs)), labels].shape

/home/hqdeng7/.conda/envs/ljy/lib/python3.11/site-packages/torch/autograd/graph.py:824: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:181.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


torch.Size([1, 269722])

In [ ]:
print(jacobs.element_size() * jacobs.nelement() / 1e6)

1.078888


In [4]:
1e2

100.0

In [27]:
num_poison = 20
not_train_indices = []
for i in range(30):
	test_indices = cls30_indices_arrs[i]
	if len(test_indices) > 8:
		tar_idx = test_indices[find_closest_samples(X_pca[cls30_labels == i], cls30_centers[i])]
		ref_input, ref_label = test_ds[tar_idx]
		fn = get_batch_myentk_fn(model100, ref_input, ref_label)
		new_not_train_indices = find_topk_samples(train_ds, fn, k=num_poison, batch_size=64, reverse=True)
		print(new_not_train_indices)
		not_train_indices = np.union1d(not_train_indices, new_not_train_indices)

 12%|█▏        | 94/782 [00:13<01:36,  7.12it/s]


KeyboardInterrupt: 

In [93]:
recover_cuda_after_oom()

13136
✅ CUDA 显存已尝试回收


In [37]:
type(model100._func_cache["params"])

tuple

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.func import functional_call, vmap, grad  # grad 在较新 torch.func 中可用
from torch.utils.data import DataLoader
import heapq
from functorch import make_functional_with_buffers

def get_param_jac(model, inputs, device="cuda"):

	model.eval()
	inputs = inputs.to(device)
	params = dict(model.named_parameters())
	buffers = dict(model.named_buffers())

	def grad_fn(params, input, class_idx):
		def loss_fn(params):
			out = functional_call(model, (params, buffers), input.unsqueeze(0))  # (1, C)
			oh = torch.zeros_like(out)  # (1, C)
			oh = oh.scatter(1, class_idx.view(1,1), 1.0)  # out-of-place, safe for vmap
			return (out[0] * oh).sum()  # 标量
		grads = grad(loss_fn)(params)
		return grads  # tuple of param-shaped grads
	
	class_idxs = torch.arange(10, device=device)

	def per_sample_jac(input):
		grads_per_class = vmap(lambda c: grad_fn(params, input, c))(class_idxs)
		# grads_per_class: tuple of tensors, each (C, *param_shape)
		flat = []
		for g in grads_per_class.values():  # over params
			flat.append(g.reshape(g.shape[0], -1))  # (C, param_dim)
		return torch.cat(flat, dim=1)  # (C, P)

	# vmap over batch
	jacobian = vmap(per_sample_jac)(inputs)  # (B, C, P)
	return jacobian


# ---------- get_batch_myentk_fn：名称、接口与原来一致，但内部使用更快的 get_param_jac（按块） ----------
def get_batch_myentk_fn(model, 
						ref_input,
						ref_label,
						loss_fn=nn.CrossEntropyLoss(reduction='none'), 
						device='cuda',
						chunk_size=32):
	model.eval()
	loss_fn = loss_fn if loss_fn.reduction == 'none' else type(loss_fn)(reduction='none')
	model = model.to(device)

	ref_input = ref_input.unsqueeze(0)
	ref_input = ref_input.to(device)

	# 用我们优化后的 get_param_jac（它内部会把 model functionalize 并缓存）
	ref_jacob = get_param_jac(model, ref_input, chunk_size=chunk_size, device=device)  # (1, out_dim, P)

	# 构造左向量（与原实现一致）
	t = torch.full((ref_jacob.shape[1],), -1, device=device)
	t[ref_label] = 1
	left_mat = ref_jacob * t.view(1, -1, 1)  # (1, out_dim, P)
	left_mat = torch.sum(left_mat.squeeze(0), 0)  # (P,)

	def batch_myentk_fn(batch):
		inputs, labels = batch
		inputs = inputs.to(device)
		labels = labels.to(device)

		# 计算 batch 的 jacob（按块）
		jacobs = get_param_jac(model, inputs, chunk_size=chunk_size, device=device)  # (B, out_dim, P)
		right_mat = jacobs[torch.arange(len(inputs)), labels]  # (B, P)

		results = right_mat.mv(left_mat)  # (B,)
		return results  # 保持 tensor 返回（跟你原来代码的返回类型一致）
	
	return batch_myentk_fn

# ---------- find_topk_samples：名字与原来一致，但内部用向量化 topk 合并以加速 ----------
def find_topk_samples(
	ds, 
	fn,
	k=16,
	batch_size=256,
	reverse=False,
	show_progress=True,
	num_workers=4,
	pin_memory=True
):
	"""
	返回 list of (value, idx) 与原来接口相同。
	优化：使用 torch.topk 在 CPU/Tensor 上流式合并 top-k，避免 Python 层 per-sample heap 操作。
	"""
	dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
	best_vals = None  # tensor
	best_idxs = None
	idx_offset = 0

	iterator = dl
	if show_progress:
		iterator = tqdm(dl)

	for batch in iterator:
		batch_res = fn(batch)  # 假设返回一维 tensor (B,) 在 device 上或 CPU 上
		if isinstance(batch_res, torch.Tensor):
			vals = batch_res.detach().cpu()
		else:
			vals = torch.tensor(batch_res)  # 兜底

		B = vals.shape[0]
		idxs = torch.arange(idx_offset, idx_offset + B, dtype=torch.long)

		# apply reverse semantics: if reverse True, we want smallest k, so invert by negating
		vals_for_sort = -vals if reverse else vals

		if best_vals is None:
			if vals_for_sort.shape[0] <= k:
				best_vals = vals_for_sort.clone()
				best_idxs = idxs.clone()
			else:
				topv, topi = torch.topk(vals_for_sort, k)
				best_vals = topv.clone()
				best_idxs = idxs[topi].clone()
		else:
			# merge existing best and this batch, keep top k
			cat_vals = torch.cat([best_vals, vals_for_sort])
			cat_idxs = torch.cat([best_idxs, idxs])
			if cat_vals.shape[0] <= k:
				best_vals = cat_vals
				best_idxs = cat_idxs
			else:
				topv, topi = torch.topk(cat_vals, k)
				best_vals = topv
				best_idxs = cat_idxs[topi]

		idx_offset += B

	if best_vals is None:
		return []

	final_vals = -best_vals if reverse else best_vals
	sorted_vals, order = torch.sort(final_vals, descending=True)
	sorted_idxs = best_idxs[order]

	return [(float(v.item()), int(i.item())) for v, i in zip(sorted_vals, sorted_idxs)]

tar_idx = test_indices[find_closest_samples(X_pca[cls30_labels == 0], cls30_centers[0])]
ref_input, ref_label = test_ds[tar_idx]
fn = get_batch_myentk_fn(model100, ref_input, ref_label)
not_train_indices = find_topk_samples(train_ds, fn, k=2, batch_size=512, reverse=True)

  0%|          | 0/98 [00:00<?, ?it/s]

100%|██████████| 98/98 [00:19<00:00,  5.12it/s]


In [70]:
list(zip(*not_train_indices))[1]

(46980, 27158)